**Status: canonical pipeline.** Lap-level podium/win probability models (`pre_model`, `live_model`), saved to `models/pre_model.json` / `models/live_model.json` and loaded by `notebooks/phase_4.ipynb`.

A separate, more elaborate stint-level exploration lives in `notebooks/phase_3_strategy_experimental.ipynb` (full pit-strategy reconstruction: compound + pit lap + win probability). It is **not** wired into `phase_4.ipynb` — treat it as a parallel research track, not a step in this pipeline.

In [ ]:
# Spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ML utilities
import numpy as np
import pandas as pd

# XGBoost
from xgboost import XGBClassifier

# Sklearn helpers
from sklearn.metrics import roc_auc_score

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"

df = spark.table(MASTER_PATH)

In [0]:
display(df)

In [0]:
print(df.columns)

In [0]:
stint_window = (
    Window
    .partitionBy("Driver", "Year", "Circuit", "Stint")
    .orderBy("LapNumber")
)

df = df.withColumn(
    "lap_delta",
    F.col("LapTime") - F.lag("LapTime").over(stint_window)
)

df = df.withColumn(
    "lap_delta",
    F.when((F.col("lap_delta") < -5) | (F.col("lap_delta") > 5), None)
     .otherwise(F.col("lap_delta"))
)

In [0]:
# 1. Race normalized lap pace
race_avg = df.groupBy("Year","Circuit","LapNumber")              .agg(F.avg("LapTime").alias("race_avg_laptime"))

df = df.join(race_avg, ["Year","Circuit","LapNumber"])

df = df.withColumn(
    "relative_laptime",
    F.col("LapTime") - F.col("race_avg_laptime")
)

# 2. Sector normalization
sector_avg = df.groupBy("Year","Circuit").agg(
    F.avg("Sector1Time").alias("sector1_avg"),
    F.avg("Sector2Time").alias("sector2_avg"),
    F.avg("Sector3Time").alias("sector3_avg")
)

df = df.join(sector_avg, ["Year","Circuit"])

df = df.withColumn("sector1_rel", F.col("Sector1Time") - F.col("sector1_avg"))
df = df.withColumn("sector2_rel", F.col("Sector2Time") - F.col("sector2_avg"))
df = df.withColumn("sector3_rel", F.col("Sector3Time") - F.col("sector3_avg"))

# 3. Overtaking ability
# This stays as a race-relative proxy and is reused in the driver profile below.
df = df.withColumn(
    "position_gain",
    F.col("QualiPosition") - F.col("FinalPosition")
)

# 4. Tyre management
stint_life = df.groupBy("Driver","Year","Stint")                .agg(F.max("TyreLife").alias("stint_life"))

tyre_management = stint_life.groupBy("Driver")                             .agg(F.avg("stint_life").alias("tyre_management"))

# 5. Driver consistency
consistency = df.groupBy("Driver")                 .agg(F.stddev("LapTime").alias("lap_consistency"))


In [0]:
# 6. Driver profile
# Build a season-normalized, car-light profile using race outcomes relative to field size
# and teammate comparisons instead of raw constructor-dominant results.
status_agg = (
    F.max(F.coalesce(F.col("Status"), F.lit(""))).alias("race_status")
    if "Status" in df.columns
    else F.first(F.lit("Finished")).alias("race_status")
)

driver_style_profile = df.groupBy("Driver").agg(
    F.avg("relative_laptime").alias("driver_pace"),
    F.avg("sector1_rel").alias("driver_sector1_skill"),
    F.avg("sector2_rel").alias("driver_sector2_skill"),
    F.avg("sector3_rel").alias("driver_sector3_skill"),
    F.avg("position_gain").alias("driver_overtake_skill"),
    F.avg("SpeedI1").alias("driver_speedI1"),
    F.avg("SpeedI2").alias("driver_speedI2"),
    F.avg("SpeedFL").alias("driver_speedFL"),
    F.avg("SpeedST").alias("driver_speedST")
)

driver_race_results = df.groupBy("Year", "Circuit", "Driver", "TeamName").agg(
    F.max("QualiPosition").alias("quali_position"),
    F.max("FinalPosition").alias("final_position"),
    F.max("LapNumber").alias("laps_completed"),
    F.max(
        F.when(
            F.upper(F.coalesce(F.col("Compound"), F.lit(""))).isin("INTERMEDIATE", "WET"),
            1
        ).otherwise(0)
    ).alias("wet_race_flag"),
    status_agg
)

event_window = Window.partitionBy("Year", "Circuit")
team_window = Window.partitionBy("Year", "Circuit", "TeamName")
field_denominator = F.when(F.col("field_size") > 1, F.col("field_size") - 1).otherwise(F.lit(1.0))
status_text = F.lower(F.col("race_status"))
classified_finish = (
    status_text.contains("finished") |
    status_text.startswith("+") |
    status_text.contains("lap")
)

driver_race_results = (
    driver_race_results
    .withColumn("field_size", F.count("*").over(event_window))
    .withColumn("team_size", F.count("*").over(team_window))
    .withColumn("race_max_laps", F.max("laps_completed").over(event_window))
    .withColumn("finish_score", 1 - ((F.col("final_position") - 1) / field_denominator))
    .withColumn("quali_score", 1 - ((F.col("quali_position") - 1) / field_denominator))
    .withColumn("quali_vs_race_delta", F.col("finish_score") - F.col("quali_score"))
    .withColumn("positions_gained_norm", (F.col("quali_position") - F.col("final_position")) / field_denominator)
    .withColumn("win_flag", (F.col("final_position") == 1).cast("int"))
    .withColumn("podium_flag", (F.col("final_position") <= 3).cast("int"))
    .withColumn(
        "dnf_flag",
        F.when(classified_finish | (F.col("laps_completed") >= 0.9 * F.col("race_max_laps")), 0).otherwise(1)
    )
)

driver_race_results = (
    driver_race_results
    .withColumn("team_finish_total", F.sum("finish_score").over(team_window))
    .withColumn("team_quali_total", F.sum("quali_score").over(team_window))
    .withColumn("team_gain_total", F.sum("positions_gained_norm").over(team_window))
    .withColumn(
        "teammate_finish_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("finish_score") - ((F.col("team_finish_total") - F.col("finish_score")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "teammate_quali_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("quali_score") - ((F.col("team_quali_total") - F.col("quali_score")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "teammate_gain_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("positions_gained_norm") - ((F.col("team_gain_total") - F.col("positions_gained_norm")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
)

driver_season_profile = driver_race_results.groupBy("Driver", "Year").agg(
    F.avg("finish_score").alias("driver_avg_finish_score_season"),
    F.avg("quali_vs_race_delta").alias("driver_quali_vs_race_delta_season"),
    F.stddev("finish_score").alias("driver_finish_consistency_season"),
    F.avg("win_flag").alias("driver_win_rate_season"),
    F.avg("podium_flag").alias("driver_podium_rate_season"),
    F.avg(F.when(F.col("wet_race_flag") == 1, F.col("finish_score"))).alias("driver_wet_finish_score_season"),
    F.avg(F.when(F.col("wet_race_flag") == 0, F.col("finish_score"))).alias("driver_dry_finish_score_season"),
    F.avg("positions_gained_norm").alias("driver_overtake_ability_season"),
    F.avg("dnf_flag").alias("driver_dnf_rate_season"),
    F.avg("teammate_finish_delta").alias("driver_teammate_finish_delta_season"),
    F.avg("teammate_quali_delta").alias("driver_teammate_quali_delta_season"),
    F.avg("teammate_gain_delta").alias("driver_teammate_gain_delta_season")
)

driver_skill_profile = driver_season_profile.groupBy("Driver").agg(
    F.avg("driver_avg_finish_score_season").alias("driver_avg_finish_score"),
    F.avg("driver_quali_vs_race_delta_season").alias("driver_quali_vs_race_delta"),
    F.avg("driver_finish_consistency_season").alias("driver_finish_consistency"),
    F.avg("driver_win_rate_season").alias("driver_win_rate"),
    F.avg("driver_podium_rate_season").alias("driver_podium_rate"),
    F.avg("driver_wet_finish_score_season").alias("driver_wet_finish_score"),
    F.avg("driver_dry_finish_score_season").alias("driver_dry_finish_score"),
    F.avg("driver_overtake_ability_season").alias("driver_overtake_ability"),
    F.avg("driver_dnf_rate_season").alias("driver_dnf_rate"),
    F.avg("driver_teammate_finish_delta_season").alias("driver_teammate_finish_delta"),
    F.avg("driver_teammate_quali_delta_season").alias("driver_teammate_quali_delta"),
    F.avg("driver_teammate_gain_delta_season").alias("driver_teammate_gain_delta")
)

driver_profile = (
    driver_style_profile
    .join(consistency, "Driver", "left")
    .join(tyre_management, "Driver", "left")
    .join(driver_skill_profile, "Driver", "left")
)

print(driver_profile.columns)


In [0]:
# 7. Constructor profile
# Capture constructor strength and reliability separately so the notebook can model car/team effects
# without mixing them into the driver profile itself.
constructor_race_results = driver_race_results.groupBy("Year", "Circuit", "TeamName").agg(
    F.avg("finish_score").alias("constructor_finish_score_race"),
    F.avg("quali_score").alias("constructor_quali_score_race"),
    F.avg("win_flag").alias("constructor_win_rate_race"),
    F.avg("podium_flag").alias("constructor_podium_rate_race"),
    F.avg("dnf_flag").alias("constructor_dnf_rate_race")
)

constructor_season_profile = constructor_race_results.groupBy("TeamName", "Year").agg(
    F.avg("constructor_finish_score_race").alias("constructor_avg_finish_score_season"),
    F.avg("constructor_quali_score_race").alias("constructor_avg_quali_score_season"),
    F.avg("constructor_win_rate_race").alias("constructor_win_rate_season"),
    F.avg("constructor_podium_rate_race").alias("constructor_podium_rate_season"),
    F.avg("constructor_dnf_rate_race").alias("constructor_dnf_rate_season")
)

constructor_profile = constructor_season_profile.groupBy("TeamName").agg(
    F.avg("constructor_avg_finish_score_season").alias("constructor_avg_finish_score"),
    F.avg("constructor_avg_quali_score_season").alias("constructor_avg_quali_score"),
    F.avg("constructor_win_rate_season").alias("constructor_win_rate"),
    F.avg("constructor_podium_rate_season").alias("constructor_podium_rate"),
    F.avg("constructor_dnf_rate_season").alias("constructor_dnf_rate"),
    F.avg("constructor_avg_finish_score_season").alias("constructor_season_strength")
)

print(constructor_profile.columns)


In [0]:
# 8. Merge driver and constructor profiles into main dataframe
df = df.join(driver_profile, "Driver", "left")
df = df.join(constructor_profile, "TeamName", "left")

display(df)


### **Step 2: Target Construction**

In [0]:
finish_window = Window.partitionBy("Year", "Circuit", "Driver")

df = df.withColumn(
    "FinalPosition",
    F.max("Position").over(finish_window)
)

In [0]:
df = (
    df
    .withColumn("PodiumFinish", (F.col("FinalPosition") <= 3).cast("int"))
    .withColumn("WinFinish", (F.col("FinalPosition") == 1).cast("int"))
)

In [0]:
weather_window = Window.partitionBy("Year", "Circuit")

df = (
    df
    .withColumn("AirTemp_delta", F.col("AirTemp_C") - F.avg("AirTemp_C").over(weather_window))
    .withColumn("TrackTemp_delta", F.col("TrackTemp_C") - F.avg("TrackTemp_C").over(weather_window))
    .withColumn("Humidity_delta", F.col("Humidity_pct") - F.avg("Humidity_pct").over(weather_window))
    .withColumn("WindSpeed_delta", F.col("WindSpeed_kmh") - F.avg("WindSpeed_kmh").over(weather_window))
)

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.default.f1_ml_lap_dataset")

### **Step 3: Feature Selection**

In [0]:
FEATURES_PRERACE = [
    "Driver",
    "TeamName",
    "Circuit",
    "Year",
    "QualiPosition",

    # # Driver style
    # 'driver_pace',
    # 'driver_sector1_skill',
    # 'driver_sector2_skill',
    # 'driver_sector3_skill',
    # 'driver_overtake_skill',
    # 'driver_speedI1',
    # 'driver_speedI2',
    # 'driver_speedFL',
    # 'driver_speedST',
    # 'lap_consistency',
    # 'tyre_management',

    # # Driver profile
    # 'driver_avg_finish_score',
    # 'driver_quali_vs_race_delta',
    # 'driver_finish_consistency',
    # 'driver_win_rate',
    # 'driver_podium_rate',
    # 'driver_wet_finish_score',
    # 'driver_dry_finish_score',
    # 'driver_overtake_ability',
    # 'driver_dnf_rate',
    # 'driver_teammate_finish_delta',
    # 'driver_teammate_quali_delta',
    # 'driver_teammate_gain_delta',

    # # Constructor profile
    # 'constructor_avg_finish_score',
    # 'constructor_avg_quali_score',
    # 'constructor_win_rate',
    # 'constructor_podium_rate',
    # 'constructor_dnf_rate',
    # 'constructor_season_strength',
]


In [0]:
FEATURES_LIVE = [
    "LapNumber",
    "RacePhase",
    "Position",
    "GapToAhead",
    "DeltaToLeader",

    "Compound",
    "TyreLife",
    "Stint",
    "TrackStatus",

    # # Driver style
    # 'driver_pace',
    # 'driver_sector1_skill',
    # 'driver_sector2_skill',
    # 'driver_sector3_skill',
    # 'driver_overtake_skill',
    # 'driver_speedI1',
    # 'driver_speedI2',
    # 'driver_speedFL',
    # 'driver_speedST',
    # 'lap_consistency',
    # 'tyre_management',

    # # Driver profile
    # 'driver_avg_finish_score',
    # 'driver_quali_vs_race_delta',
    # 'driver_finish_consistency',
    # 'driver_win_rate',
    # 'driver_podium_rate',
    # 'driver_wet_finish_score',
    # 'driver_dry_finish_score',
    # 'driver_overtake_ability',
    # 'driver_dnf_rate',
    # 'driver_teammate_finish_delta',
    # 'driver_teammate_quali_delta',
    # 'driver_teammate_gain_delta',

    # # Constructor profile
    # 'constructor_avg_finish_score',
    # 'constructor_avg_quali_score',
    # 'constructor_win_rate',
    # 'constructor_podium_rate',
    # 'constructor_dnf_rate',
    # 'constructor_season_strength',

    # Weather (relative only)
    "AirTemp_delta",
    "TrackTemp_delta",
    "Humidity_delta",
    "WindSpeed_delta"
]


### **Step 4: Convert to Pandas**

In [ ]:
df_prerace = df.select(FEATURES_PRERACE + ["PodiumFinish"])
# "Year" isn't in FEATURES_LIVE (it's not a feature the live model trains on)
# but we need it to apply the same year-based train/val/test split used for
# the pre-race model, instead of a random per-lap split.
df_live = df.select(FEATURES_LIVE + ["Year", "PodiumFinish"])

pdf_prerace = df_prerace.toPandas()
pdf_live = df_live.toPandas()

In [0]:
CATEGORICAL_PRERACE = ["Driver", "TeamName", "Circuit"]
CATEGORICAL_LIVE = ["Compound", "RacePhase"]

for c in CATEGORICAL_PRERACE:
    pdf_prerace[c] = pdf_prerace[c].astype("category")

for c in CATEGORICAL_LIVE:
    pdf_live[c] = pdf_live[c].astype("category")

### **Step 6: Train Pre-Race Strategy Probability Model**

In [0]:
# Sort by year just to be safe
pdf_prerace = pdf_prerace.sort_values("Year")

X_pre = pdf_prerace[FEATURES_PRERACE]
y_pre = pdf_prerace["PodiumFinish"]

# ---------# Time-based split
# ---------
TRAIN_END_YEAR = 2020
VAL_YEAR = 2021
TEST_START_YEAR = 2022

# Train
train_mask = pdf_prerace["Year"] <= TRAIN_END_YEAR

# Validation (optional but recommended for tuning)
val_mask = pdf_prerace["Year"] == VAL_YEAR

# Test (future simulation)
test_mask = pdf_prerace["Year"] >= TEST_START_YEAR

X_pre_train = X_pre.loc[train_mask].copy()
y_pre_train = y_pre.loc[train_mask].copy()

X_pre_val = X_pre.loc[val_mask].copy()
y_pre_val = y_pre.loc[val_mask].copy()

X_pre_test = X_pre.loc[test_mask].copy()
y_pre_test = y_pre.loc[test_mask].copy()

In [0]:
pdf_prerace.head()

In [ ]:
pre_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    early_stopping_rounds=30,
)

# X_pre_val/y_pre_val (the 2021 season) were being carved out and never used.
# Wiring them as eval_set lets XGBoost stop boosting once the held-out year
# stops improving, instead of always training all 400 rounds.
pre_model.fit(
    X_pre_train, y_pre_train,
    eval_set=[(X_pre_val, y_pre_val)],
    verbose=False,
)


In [0]:
from sklearn.metrics import roc_auc_score, classification_report

probs = pre_model.predict_proba(X_pre_test)[:, 1]
auc = roc_auc_score(y_pre_test, probs)

print("Pre-race AUC (unseen races):", round(auc, 4))
#print(classification_report(y_test, pre_model.predict(X_test)))

### **Step 5: Train Live Podium Probability Model**

In [ ]:
# Same year-based split as the pre-race model (TRAIN_END_YEAR / VAL_YEAR /
# TEST_START_YEAR, defined above), instead of:
#   groups = pdf_live.index; GroupShuffleSplit(...).split(pdf_live, groups=groups)
# Using each row's own index as its "group" made GroupShuffleSplit degenerate
# into a plain random split: it ignored year entirely and let laps from the
# same race land on both sides of the split. Consecutive laps are heavily
# autocorrelated (position/gap/compound barely change lap-to-lap), so that
# was inflating the reported AUC with near-duplicate train/test rows.
pdf_live = pdf_live.sort_values("Year")

train_mask_live = pdf_live["Year"] <= TRAIN_END_YEAR
val_mask_live = pdf_live["Year"] == VAL_YEAR
test_mask_live = pdf_live["Year"] >= TEST_START_YEAR

X_train = pdf_live.loc[train_mask_live, FEATURES_LIVE].copy()
y_train = pdf_live.loc[train_mask_live, "PodiumFinish"].copy()

X_val_live = pdf_live.loc[val_mask_live, FEATURES_LIVE].copy()
y_val_live = pdf_live.loc[val_mask_live, "PodiumFinish"].copy()

X_test = pdf_live.loc[test_mask_live, FEATURES_LIVE].copy()
y_test = pdf_live.loc[test_mask_live, "PodiumFinish"].copy()

live_model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=30,
)

live_model.fit(
    X_train, y_train,
    eval_set=[(X_val_live, y_val_live)],
    verbose=False,
)


In [0]:
probs = live_model.predict_proba(X_test)[:, 1]
print("LIVE MODEL AUC:", roc_auc_score(y_test, probs))

In [0]:
baseline_podium = pre_model.predict_proba(X_pre.iloc[[0]])[0][1]
live_podium = live_model.predict_proba(X_test.iloc[[0]])[0][1]

delta = live_podium - baseline_podium
print(delta)

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(pre_model, max_num_features=15)
plt.show()

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(live_model, max_num_features=15)
plt.show()

In [0]:
probs = pre_model.predict_proba(X_pre)[:, 1]
roc_auc_score(y_pre, probs)

In [0]:
from sklearn.calibration import calibration_curve

# predicted probabilities
probs = pre_model.predict_proba(X_pre)[:, 1]

# calibration data
prob_true, prob_pred = calibration_curve(
    y_pre,
    probs,
    n_bins=10,
    strategy="uniform"
)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

# Perfect calibration reference
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

# Your model
plt.plot(prob_pred, prob_true, marker="o", label="Pre-race model")

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration curve (Pre-race model)")
plt.legend()
plt.grid(True)

plt.show()

In [0]:
live_probs = live_model.predict_proba(X_test)[:, 1]

prob_true_l, prob_pred_l = calibration_curve(
    y_test,
    live_probs,
    n_bins=10
)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--")
plt.plot(prob_pred_l, prob_true_l, marker="o")
plt.title("Calibration curve (Live model)")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.grid(True)
plt.show()


In [0]:
# Save Model to local filesystem only (serverless clusters cannot write to /dbfs/FileStore)
pre_model.save_model("../models/pre_model.json")
live_model.save_model("../models/live_model.json")

# Note: Copying to /dbfs/FileStore/models/ is not supported on serverless clusters. Models are saved in /models and can be downloaded from there if needed.

In [0]:
### Trial

# Convert profile tables to pandas lookup tables for quick local inference helpers.
driver_profile_pdf = driver_profile.toPandas().set_index("Driver")
constructor_profile_pdf = constructor_profile.toPandas().set_index("TeamName")

DRIVER_PROFILE_FEATURES = [
    "driver_pace",
    "driver_sector1_skill",
    "driver_sector2_skill",
    "driver_sector3_skill",
    "driver_overtake_skill",
    "driver_speedI1",
    "driver_speedI2",
    "driver_speedFL",
    "driver_speedST",
    "lap_consistency",
    "tyre_management",
    "driver_avg_finish_score",
    "driver_quali_vs_race_delta",
    "driver_finish_consistency",
    "driver_win_rate",
    "driver_podium_rate",
    "driver_wet_finish_score",
    "driver_dry_finish_score",
    "driver_overtake_ability",
    "driver_dnf_rate",
    "driver_teammate_finish_delta",
    "driver_teammate_quali_delta",
    "driver_teammate_gain_delta",
]

CONSTRUCTOR_PROFILE_FEATURES = [
    "constructor_avg_finish_score",
    "constructor_avg_quali_score",
    "constructor_win_rate",
    "constructor_podium_rate",
    "constructor_dnf_rate",
    "constructor_season_strength",
]


def _fallback_profile(pdf, index_name):
    numeric_profile = pdf.reset_index().drop(columns=[index_name]).mean(numeric_only=True)
    return numeric_profile.to_dict()


def predict_prerace(driver, team, circuit, year, quali_pos):
    if driver in driver_profile_pdf.index:
        driver_values = driver_profile_pdf.loc[driver].to_dict()
    else:
        driver_values = _fallback_profile(driver_profile_pdf, "Driver")

    if team in constructor_profile_pdf.index:
        constructor_values = constructor_profile_pdf.loc[team].to_dict()
    else:
        constructor_values = _fallback_profile(constructor_profile_pdf, "TeamName")

    row = {
        "Driver": driver,
        "TeamName": team,
        "Circuit": circuit,
        "Year": year,
        "QualiPosition": quali_pos,
    }

    for feature in DRIVER_PROFILE_FEATURES:
        row[feature] = driver_values.get(feature, np.nan)

    for feature in CONSTRUCTOR_PROFILE_FEATURES:
        row[feature] = constructor_values.get(feature, np.nan)

    row = pd.DataFrame([row])

    for c in CATEGORICAL_PRERACE:
        row[c] = row[c].astype("category")

    prob = pre_model.predict_proba(row[FEATURES_PRERACE])[0][1]
    return prob


In [0]:
l = list(pdf_prerace["TeamName"].unique())
print(l)

In [0]:
l = list(pdf_prerace["Circuit"].unique())
print(l)

In [0]:
probability = predict_prerace(
    driver="HAM",
    team="Mercedes",
    circuit="Silverstone",
    year=2026,
    quali_pos=1
) * 100

print(f"{probability:.2f}%")

In [0]:
for driver in pdf_prerace["Driver"].unique():
    probability = predict_prerace(
        driver=driver,
        team="Mercedes",
        circuit="Silverstone",
        year=2023,
        quali_pos=5
    ) * 100

    print(f"{driver} - {probability:.2f}%")